# run_evaluation.ipynb

Batch LLM-as-judge evaluation across all simulation runs. Run **after** `run_simulation.ipynb`.

**What this does:**
1. Loads all JSONL run files from `outputs/runs/`
2. Judges each agent decision (per-intervention) on 3 criteria (1–5)
3. Judges each agent's full simulation trajectory holistically (3 criteria, 1–5)
4. Exports results to a 3-tab Excel workbook in `outputs/eval/`

**Tabs in the output:**
- `Interventions` — per-event scores + notes for every agent × run
- `Full Simulation` — holistic scores + notes per agent × run
- `Cost & Latency` — agent model cost, judge cost, sim latency per run

**Judge model:** `claude-opus-4-6` at temperature=0 (deterministic).

See `System Design and Notes/evaluation_plan.md` for full design rationale.

---
## Section 0 — Install & imports

In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic openai \
  sentence-transformers \
  'numpy>=1.26' \
  'pandas>=2.2' \
  openpyxl \
  pyyaml

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/Spring 2026/INFO 290: GenAI/GenAI Project 290/berkeley-homes-wildfire-agent-simulation'
except ImportError:
    PROJECT_PATH = os.path.abspath('..')   # running locally: one level up from notebooks/

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import json
import yaml
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import pandas as pd

sys.path.insert(0, PROJECT_PATH)

from src.llm.client import (
    Config, init_clients, init_openrouter_client,
    usage_tracker, judge_intervention, judge_full_simulation,
    judge_intervention_v2, judge_full_simulation_v2,
    judge_held_out,
)

print('Imports OK')

---
## Section 1 — Config

**Only this cell needs to change between runs.**

- `RUNS_DIR`: where simulation JSONLs are stored
- `AGENT_YAML_DIR`: where the selected agent YAMLs live
- `RUN_LABELS_TO_EVAL`: set to `None` to evaluate all runs found, or list specific labels
- `EXPORT_FILENAME`: name of the output Excel file

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
RUNS_DIR       = Path('outputs/runs')
AGENT_YAML_DIR = Path('config/agents/selected')
EVAL_DIR       = Path('outputs/eval')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_FILENAME = EVAL_DIR / f'evaluation_{timestamp}.xlsx'

# ── Filter: which run labels to evaluate ──────────────────────────────────────
# Set to None to evaluate everything found in RUNS_DIR.
# Or list specific labels, e.g.:
#   RUN_LABELS_TO_EVAL = ['Premium_Full', 'Baseline_Full']
RUN_LABELS_TO_EVAL = None

# ── Judge model ────────────────────────────────────────────────────────────────
JUDGE_MODEL = 'claude-opus-4-6'

print(f'Runs dir:    {RUNS_DIR}')
print(f'Agents dir:  {AGENT_YAML_DIR}')
print(f'Export path: {EXPORT_FILENAME}')
print(f'Judge model: {JUDGE_MODEL}')

---
## Section 2 — Connect to APIs

In [ ]:
client_anthropic = init_clients()

# Judge config — deterministic, strong model
judge_config = Config(JUDGE_MODEL=JUDGE_MODEL, JUDGE_TEMPERATURE=0.0)

---
## Section 3 — Load simulation results

Scans `outputs/runs/` for JSONL files, reads the `run_config` entry from each to
identify the run label and decision model, then loads the agent YAMLs from
`config/agents/selected/` to supply memory seeds for the judge.

In [ ]:
def load_agent_yaml(yaml_path: Path) -> dict:
    """Load agent config — handles flat and agents-list-wrapped formats."""
    with open(yaml_path, encoding='utf-8') as f:
        raw = yaml.safe_load(f)
    if 'agents' in raw and isinstance(raw['agents'], list):
        return raw['agents'][0]
    return raw

# Load all selected agent YAMLs keyed by agent id
agent_configs = {}
for yaml_path in sorted(AGENT_YAML_DIR.glob('*.yaml')):
    cfg = load_agent_yaml(yaml_path)
    agent_id = cfg['id']
    agent_configs[agent_id] = cfg
    print(f'  Loaded agent: {agent_id} ({cfg["display_name"]})')

print(f'\n{len(agent_configs)} agent configs loaded.')

In [ ]:
def parse_jsonl(path: Path) -> list:
    entries = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

# Group JSONL entries by run_label
# Structure: {run_label: {run_config: {...}, decisions: [...], run_summary: {...}}}
runs = {}

jsonl_files = sorted(RUNS_DIR.glob('*.jsonl'))
print(f'Found {len(jsonl_files)} JSONL files in {RUNS_DIR}\n')

for path in jsonl_files:
    entries = parse_jsonl(path)
    if not entries:
        continue

    run_config = next((e for e in entries if e.get('entry_type') == 'run_config'), None)
    if not run_config:
        print(f'  SKIP {path.name} — no run_config entry found')
        continue

    run_label = run_config.get('run_label', path.stem)

    if RUN_LABELS_TO_EVAL and run_label not in RUN_LABELS_TO_EVAL:
        print(f'  SKIP {path.name} — label {run_label!r} not in RUN_LABELS_TO_EVAL')
        continue

    decisions  = [e for e in entries if e.get('entry_type') == 'decision']
    run_summary = next((e for e in entries if e.get('entry_type') == 'run_summary'), {})

    runs[run_label] = {
        'run_config':  run_config,
        'decisions':   decisions,
        'run_summary': run_summary,
        'source_file': path.name,
    }
    print(f'  Loaded {path.name} → label={run_label!r}, {len(decisions)} decision entries')

print(f'\n{len(runs)} runs ready for evaluation.')

---
## Section 4 — Per-intervention judging

One judge call per agent decision entry. Judge receives:
- Full `seed_narrative` (from agent YAML)
- All memory seeds (from agent YAML, descriptions only)
- Intervention text, decision, reasoning (from JSONL)

Scores 3 criteria (1–5) with a 1–2 sentence note each.

In [ ]:
usage_tracker.reset()   # track judge tokens separately from simulation tokens
judge_start = time.time()

intervention_results = []   # one dict per row → Tab 1

for run_label, run_data in runs.items():
    print(f'\n{"="*60}')
    print(f'Run: {run_label}  ({len(run_data["decisions"])} decisions)')
    print(f'{"="*60}')

    for entry in run_data['decisions']:
        agent_id      = entry['agent_id']
        display_name  = entry['agent_display_name']
        event_type    = entry['event_type']
        day           = entry['tick']
        intervention  = entry['intervention']
        decision      = entry['decision']
        reasoning     = entry['reasoning']

        # Fetch agent background from YAML
        agent_cfg      = agent_configs.get(agent_id, {})
        seed_narrative = agent_cfg.get('seed_narrative', entry.get('seed_personality', ''))
        memory_seeds   = agent_cfg.get('memory_seeds', [])

        print(f'  Judging: {display_name} / Day {day} {event_type}...', end=' ', flush=True)

        scores = judge_intervention(
            client_anthropic=client_anthropic,
            config=judge_config,
            seed_narrative=seed_narrative,
            memory_seeds=memory_seeds,
            intervention=intervention,
            decision=decision,
            reasoning=reasoning,
        )

        bp  = scores.get('behavioral_plausibility')
        pc  = scores.get('persona_consistency')
        ir  = scores.get('intervention_responsiveness')
        print(f'BP={bp} PC={pc} IR={ir}')

        intervention_results.append({
            'run_label':                  run_label,
            'agent_id':                   agent_id,
            'agent_display_name':         display_name,
            'day':                        day,
            'event_type':                 event_type,
            'intervention':               intervention,
            'decision':                   decision,
            'reasoning':                  reasoning,
            'behavioral_plausibility':    bp,
            'bp_note':                    scores.get('note_plausibility'),
            'persona_consistency':        pc,
            'pc_note':                    scores.get('note_consistency'),
            'intervention_responsiveness': ir,
            'ir_note':                    scores.get('note_responsiveness'),
        })

judge_elapsed = time.time() - judge_start
print(f'\nPer-intervention judging complete. {len(intervention_results)} rows.')
print(f'Elapsed: {judge_elapsed:.1f}s')

---
## Section 5 — Full simulation judging

One judge call per agent per run. Judge sees the **complete trajectory** —
all interventions in chronological order — and gives a holistic 1–5 score
for each criterion. This is not an average of per-intervention scores.

In [ ]:
simulation_results = []   # one dict per row → Tab 2

for run_label, run_data in runs.items():
    print(f'\n{"="*60}')
    print(f'Run: {run_label} — full simulation judging')
    print(f'{"="*60}')

    # Group decisions by agent
    by_agent = defaultdict(list)
    for entry in sorted(run_data['decisions'], key=lambda e: e['tick']):
        by_agent[entry['agent_id']].append(entry)

    for agent_id, entries in by_agent.items():
        display_name = entries[0]['agent_display_name']

        agent_cfg      = agent_configs.get(agent_id, {})
        seed_narrative = agent_cfg.get('seed_narrative', entries[0].get('seed_personality', ''))
        memory_seeds   = agent_cfg.get('memory_seeds', [])

        # Build trajectory list for the judge
        all_decisions = [
            {
                'day':          e['tick'],
                'event_type':   e['event_type'],
                'intervention': e['intervention'],
                'decision':     e['decision'],
                'reasoning':    e['reasoning'],
            }
            for e in entries
        ]

        print(f'  Judging: {display_name} ({len(all_decisions)} events)...', end=' ', flush=True)

        scores = judge_full_simulation(
            client_anthropic=client_anthropic,
            config=judge_config,
            seed_narrative=seed_narrative,
            memory_seeds=memory_seeds,
            all_decisions=all_decisions,
        )

        bp  = scores.get('behavioral_plausibility')
        pc  = scores.get('persona_consistency')
        ir  = scores.get('intervention_responsiveness')
        print(f'BP={bp} PC={pc} IR={ir}')

        simulation_results.append({
            'run_label':                   run_label,
            'agent_id':                    agent_id,
            'agent_display_name':          display_name,
            'behavioral_plausibility':     bp,
            'bp_note':                     scores.get('note_plausibility'),
            'persona_consistency':         pc,
            'pc_note':                     scores.get('note_consistency'),
            'intervention_responsiveness': ir,
            'ir_note':                     scores.get('note_responsiveness'),
        })

print(f'\nFull simulation judging complete. {len(simulation_results)} rows.')

---
## Section 6 — Build cost & latency table

Combines:
- Agent model cost + simulation latency from `run_summary` in each JSONL
- Judge model cost from `usage_tracker` accumulated during this evaluation session

In [ ]:
# Judge cost split evenly across runs (total judge tokens ÷ number of runs)
judge_usage = usage_tracker.to_dict(agent_model='claude-opus-4-6', judge_model=JUDGE_MODEL)
n_runs = len(runs)
judge_cost_per_run = judge_usage['judge_cost_usd'] / n_runs if n_runs else 0
judge_in_per_run   = judge_usage['judge_tokens_in']  // n_runs if n_runs else 0
judge_out_per_run  = judge_usage['judge_tokens_out'] // n_runs if n_runs else 0

cost_rows = []
for run_label, run_data in runs.items():
    rs = run_data.get('run_summary', {})
    rc = run_data['run_config']

    agent_cost    = rs.get('agent_cost_usd', None)
    total_cost    = (agent_cost or 0) + judge_cost_per_run

    cost_rows.append({
        'run_label':            run_label,
        'agent_model':          rc.get('decision_model', ''),
        'agent_tokens_in':      rs.get('agent_tokens_in', ''),
        'agent_tokens_out':     rs.get('agent_tokens_out', ''),
        'agent_cost_usd':       agent_cost,
        'judge_tokens_in':      judge_in_per_run,
        'judge_tokens_out':     judge_out_per_run,
        'judge_cost_usd':       round(judge_cost_per_run, 6),
        'total_cost_usd':       round(total_cost, 6) if agent_cost is not None else '',
        'run_latency_seconds':  rs.get('latency_seconds', ''),
    })

print('Cost & latency summary:')
for r in cost_rows:
    print(f"  {r['run_label']:<45} agent=${r['agent_cost_usd']}  "
          f"judge=${r['judge_cost_usd']}  latency={r['run_latency_seconds']}s")

---
## Section 7 — Export to Excel

Writes a 3-tab Excel workbook to `outputs/eval/`.

In [ ]:
df_interventions = pd.DataFrame(intervention_results)
df_simulations   = pd.DataFrame(simulation_results)
df_costs         = pd.DataFrame(cost_rows)

SCORE_COLS = ['behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness']

with pd.ExcelWriter(EXPORT_FILENAME, engine='openpyxl') as writer:
    df_interventions.to_excel(writer, sheet_name='Interventions',   index=False)
    df_simulations.to_excel(writer,   sheet_name='Full Simulation', index=False)
    df_costs.to_excel(writer,         sheet_name='Cost & Latency',  index=False)

    for sheet_name, df in [('Interventions', df_interventions), ('Full Simulation', df_simulations)]:
        ws = writer.sheets[sheet_name]
        # Format score columns as 2dp
        for col_idx, col_name in enumerate(df.columns, start=1):
            if col_name in SCORE_COLS:
                for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                    for cell in row:
                        cell.number_format = '0.00'

    # Auto-size columns for readability
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0 for cell in col),
                default=10
            )
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 80)

print(f'\nExported to: {EXPORT_FILENAME}')
print(f'  Interventions tab: {len(df_interventions)} rows')
print(f'  Full Simulation tab: {len(df_simulations)} rows')
print(f'  Cost & Latency tab: {len(df_costs)} rows')

---
## Section 8 — Quick summary

Print score averages per run label and per agent for a quick read.

In [ ]:
print('PER-INTERVENTION AVERAGES BY RUN')
print('=' * 60)
if not df_interventions.empty:
    grp = df_interventions.groupby('run_label')[[
        'behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness'
    ]].mean().round(2)
    print(grp.to_string())

print('\nFULL SIMULATION SCORES BY RUN × AGENT')
print('=' * 60)
if not df_simulations.empty:
    print(df_simulations[[
        'run_label', 'agent_display_name',
        'behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness'
    ]].to_string(index=False))

---
## Section 9 — Judge v2: 1-10 scale, evidence-anchored, seed-narrative only

Runs the updated judge prompt on the same loaded data. Key differences from v1:
- **1-10 integer scale** (no half-points)
- **Seed narrative only** — memory seeds not passed to judge, so its context matches what No_Memory agents actually had
- **Evidence-anchored rubric** — each band requires verifiable evidence; judge cannot score above 6 without citing specific evidence
- **Social desirability / trait neutralisation warnings** explicit in each criterion
- **Calibrated centre** — 5-6 is the expected range for plausible but generic responses

Exports to a separate `_v2` Excel file for direct comparison with v1.

In [ ]:
usage_tracker.reset()
judge_start_v2 = time.time()

intervention_results_v2 = []

for run_label, run_data in runs.items():
    print(f'\n{"="*60}')
    print(f'Run: {run_label}  ({len(run_data["decisions"])} decisions)')
    print(f'{"="*60}')

    for entry in run_data['decisions']:
        agent_id      = entry['agent_id']
        display_name  = entry['agent_display_name']
        event_type    = entry['event_type']
        day           = entry['tick']
        intervention  = entry['intervention']
        decision      = entry['decision']
        reasoning     = entry['reasoning']

        agent_cfg      = agent_configs.get(agent_id, {})
        seed_narrative = agent_cfg.get('seed_narrative', entry.get('seed_personality', ''))

        print(f'  Judging: {display_name} / Day {day} {event_type}...', end=' ', flush=True)

        scores = judge_intervention_v2(
            client_anthropic=client_anthropic,
            config=judge_config,
            seed_narrative=seed_narrative,
            memory_seeds=[],   # not passed to v2 judge
            intervention=intervention,
            decision=decision,
            reasoning=reasoning,
        )

        bp  = scores.get('behavioral_plausibility')
        pc  = scores.get('persona_consistency')
        ir  = scores.get('intervention_responsiveness')
        print(f'BP={bp} PC={pc} IR={ir}')

        intervention_results_v2.append({
            'run_label':                   run_label,
            'agent_id':                    agent_id,
            'agent_display_name':          display_name,
            'day':                         day,
            'event_type':                  event_type,
            'intervention':                intervention,
            'decision':                    decision,
            'reasoning':                   reasoning,
            'behavioral_plausibility':     bp,
            'bp_note':                     scores.get('note_plausibility'),
            'persona_consistency':         pc,
            'pc_note':                     scores.get('note_consistency'),
            'intervention_responsiveness': ir,
            'ir_note':                     scores.get('note_responsiveness'),
        })

judge_elapsed_v2 = time.time() - judge_start_v2
print(f'\nv2 per-intervention judging complete. {len(intervention_results_v2)} rows.')
print(f'Elapsed: {judge_elapsed_v2:.1f}s')

In [ ]:
simulation_results_v2 = []

for run_label, run_data in runs.items():
    print(f'\n{"="*60}')
    print(f'Run: {run_label} — full simulation judging (v2)')
    print(f'{"="*60}')

    by_agent = defaultdict(list)
    for entry in sorted(run_data['decisions'], key=lambda e: e['tick']):
        by_agent[entry['agent_id']].append(entry)

    for agent_id, entries in by_agent.items():
        display_name   = entries[0]['agent_display_name']
        agent_cfg      = agent_configs.get(agent_id, {})
        seed_narrative = agent_cfg.get('seed_narrative', entries[0].get('seed_personality', ''))

        all_decisions = [
            {
                'day':          e['tick'],
                'event_type':   e['event_type'],
                'intervention': e['intervention'],
                'decision':     e['decision'],
                'reasoning':    e['reasoning'],
            }
            for e in entries
        ]

        print(f'  Judging: {display_name} ({len(all_decisions)} events)...', end=' ', flush=True)

        scores = judge_full_simulation_v2(
            client_anthropic=client_anthropic,
            config=judge_config,
            seed_narrative=seed_narrative,
            memory_seeds=[],   # not passed to v2 judge
            all_decisions=all_decisions,
        )

        bp  = scores.get('behavioral_plausibility')
        pc  = scores.get('persona_consistency')
        ir  = scores.get('intervention_responsiveness')
        print(f'BP={bp} PC={pc} IR={ir}')

        simulation_results_v2.append({
            'run_label':                   run_label,
            'agent_id':                    agent_id,
            'agent_display_name':          display_name,
            'behavioral_plausibility':     bp,
            'bp_note':                     scores.get('note_plausibility'),
            'persona_consistency':         pc,
            'pc_note':                     scores.get('note_consistency'),
            'intervention_responsiveness': ir,
            'ir_note':                     scores.get('note_responsiveness'),
        })

print(f'\nv2 full simulation judging complete. {len(simulation_results_v2)} rows.')

In [ ]:
df_interventions_v2 = pd.DataFrame(intervention_results_v2)
df_simulations_v2   = pd.DataFrame(simulation_results_v2)

EXPORT_FILENAME_V2 = EVAL_DIR / f'evaluation_v2_{timestamp}.xlsx'
SCORE_COLS_V2 = ['behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness']

with pd.ExcelWriter(EXPORT_FILENAME_V2, engine='openpyxl') as writer:
    df_interventions_v2.to_excel(writer, sheet_name='Interventions',   index=False)
    df_simulations_v2.to_excel(writer,   sheet_name='Full Simulation', index=False)
    df_costs.to_excel(writer,            sheet_name='Cost & Latency',  index=False)

    # Auto-size columns
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0 for cell in col),
                default=10
            )
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 80)

print(f'\nExported to: {EXPORT_FILENAME_V2}')
print(f'  Interventions tab: {len(df_interventions_v2)} rows')
print(f'  Full Simulation tab: {len(df_simulations_v2)} rows')

In [ ]:
SCORE_COLS = ['behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness']

print('V2 PER-INTERVENTION AVERAGES BY RUN (1-10 scale)')
print('=' * 60)
if not df_interventions_v2.empty:
    grp = df_interventions_v2.groupby('run_label')[SCORE_COLS].mean().round(2)
    print(grp.to_string())

print('\nV1 PER-INTERVENTION AVERAGES BY RUN (1-5 scale, for comparison)')
print('=' * 60)
if not df_interventions.empty:
    grp_v1 = df_interventions.groupby('run_label')[SCORE_COLS].mean().round(2)
    print(grp_v1.to_string())

print('\nV2 FULL SIMULATION SCORES BY RUN × AGENT')
print('=' * 60)
if not df_simulations_v2.empty:
    print(df_simulations_v2[['run_label', 'agent_display_name'] + SCORE_COLS].to_string(index=False))

---
## Section 10 — Held-out fidelity judging

Compares simulated agent responses against real interview responses for the same scenario.
Only runs for agent × event combinations where a real held-out response exists in the agent YAML.

**Available held-outs:** 18 responses across 5 agents and up to 5 event types.

**Three criteria (1-10):**
- `thematic_alignment` — same underlying concerns engaged
- `stance_alignment` — same attitude / position toward the situation
- `voice_alignment` — similar tone, hedging, register

Exports to `evaluation_held_out_<timestamp>.xlsx`.

In [ ]:
# Build held-out lookup: {agent_id: {event_type: {real_response, context}}}
held_out_lookup = {}
for agent_id, cfg in agent_configs.items():
    held = cfg.get('held_out_responses', {})
    if held:
        held_out_lookup[agent_id] = {
            evt: val for evt, val in held.items()
            if val.get('real_response') is not None
        }

total_available = sum(len(v) for v in held_out_lookup.values())
print(f'Held-out responses available: {total_available}')
for agent_id, evts in held_out_lookup.items():
    name = agent_configs[agent_id]['display_name']
    print(f'  {name}: {list(evts.keys())}')

usage_tracker.reset()
judge_start_ho = time.time()

held_out_results = []

for run_label, run_data in runs.items():
    print(f'\n{"="*60}')
    print(f'Run: {run_label}')
    print(f'{"="*60}')

    for entry in run_data['decisions']:
        agent_id   = entry['agent_id']
        event_type = entry['event_type']

        # Skip if no held-out for this agent × event
        held = held_out_lookup.get(agent_id, {}).get(event_type)
        if not held:
            continue

        display_name   = entry['agent_display_name']
        day            = entry['tick']
        agent_cfg      = agent_configs.get(agent_id, {})
        seed_narrative = agent_cfg.get('seed_narrative', entry.get('seed_personality', ''))

        print(f'  {display_name} / {event_type}...', end=' ', flush=True)

        scores = judge_held_out(
            client_anthropic=client_anthropic,
            config=judge_config,
            seed_narrative=seed_narrative,
            event_type=event_type,
            real_response=held['real_response'],
            real_context=held.get('context', ''),
            simulated_decision=entry['decision'],
            simulated_reasoning=entry['reasoning'],
        )

        ta = scores.get('thematic_alignment')
        sa = scores.get('stance_alignment')
        va = scores.get('voice_alignment')
        print(f'TA={ta} SA={sa} VA={va}')

        held_out_results.append({
            'run_label':           run_label,
            'agent_id':            agent_id,
            'agent_display_name':  display_name,
            'day':                 day,
            'event_type':          event_type,
            'real_response':       held['real_response'],
            'real_context':        held.get('context', ''),
            'simulated_decision':  entry['decision'],
            'simulated_reasoning': entry['reasoning'],
            'thematic_alignment':  ta,
            'thematic_note':       scores.get('note_thematic'),
            'stance_alignment':    sa,
            'stance_note':         scores.get('note_stance'),
            'voice_alignment':     va,
            'voice_note':          scores.get('note_voice'),
        })

judge_elapsed_ho = time.time() - judge_start_ho
print(f'\nHeld-out judging complete. {len(held_out_results)} rows.')
print(f'Elapsed: {judge_elapsed_ho:.1f}s')

In [ ]:
df_held_out = pd.DataFrame(held_out_results)

EXPORT_FILENAME_HO = EVAL_DIR / f'evaluation_held_out_{timestamp}.xlsx'
HO_SCORE_COLS = ['thematic_alignment', 'stance_alignment', 'voice_alignment']

with pd.ExcelWriter(EXPORT_FILENAME_HO, engine='openpyxl') as writer:
    df_held_out.to_excel(writer, sheet_name='Held-Out Fidelity', index=False)

    ws = writer.sheets['Held-Out Fidelity']
    for col in ws.columns:
        max_len = max(
            (len(str(cell.value)) if cell.value is not None else 0 for cell in col),
            default=10
        )
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 80)

print(f'Exported to: {EXPORT_FILENAME_HO}')
print(f'  Held-Out Fidelity tab: {len(df_held_out)} rows')

# Quick summary
if not df_held_out.empty:
    print('\nMEAN SCORES BY RUN')
    print('=' * 60)
    print(df_held_out.groupby('run_label')[HO_SCORE_COLS].mean().round(2).to_string())
    print('\nMEAN SCORES BY AGENT (across all runs)')
    print('=' * 60)
    print(df_held_out.groupby('agent_display_name')[HO_SCORE_COLS].mean().round(2).to_string())